# <u>Project work, part 4 - Machine Learning<u>

**Here are the links to Streamlit and Github:**
- Streamlit URL: https://ind320-rajvir-app-repo-cmhz46bwwk9apw8zvbaxa2.streamlit.app/
- Github URL: https://github.com/rajern/ind320-rajvir

# Tasks

## Jupyter Notebook
- Use the Elhub API to retrieve hourly production data for all price areas using PRODUCTION_PER_GROUP_MBA_HOUR for all days and hours of the years 2022 - 2024.
    - Handle these data the same way as in part 2 of the project, appending the new data after the 2021 data, both in Cassandra (using Spark - see updated advice in the installation page if you struggle) and MongoDB.
    - Using new tables in your databases, but otherwise the same strategy, retrieve hourly consumption data for all price areas using CONSUMPTION_PER_GROUP_MBA_HOUR for all days and hours of the years 2021 - 2024.
    - If you have example data in MongoDB, this may need to be removed to have room for the data above. 
- Remember to fill in the log and AI mentioned in the General section above.

In [10]:
import os, sys

# Add environment variables
os.environ["JAVA_HOME"] = r"C:\Users\rajvi\AppData\Local\Programs\Microsoft\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Ensure Spark uses the Python interpreter from the kernel
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Add Java and winutils to PATH
os.environ["PATH"] = rf"{os.environ['HADOOP_HOME']}\bin;{os.environ['JAVA_HOME']}\bin;" + os.environ["PATH"]

In [11]:
# Install Cassandra driver
!pip install -q cassandra-driver

In [12]:
# Connect to local Cassandra container
from cassandra.cluster import Cluster
# cluster = Cluster(['localhost'], port=9042)
cluster = Cluster(['127.0.0.1'], port=9042)
session = cluster.connect()

In [13]:
# Start Spark with connector to Cassandra
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("SparkCassandraApp")
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1")
    .config("spark.sql.extensions", "com.datastax.spark.connector.CassandraSparkExtensions")
    .config("spark.sql.catalog.mycatalog", "com.datastax.spark.connector.datasource.CassandraCatalog")
    # .config("spark.cassandra.connection.host", "localhost")
    .config("spark.cassandra.connection.host", "127.0.0.1")
    .config("spark.cassandra.connection.port", "9042")
    .getOrCreate()
)

In [14]:
# Read MongoDG URI
from pathlib import Path
import tomllib

secrets_path = Path("../.streamlit/secrets.toml")
uri = tomllib.load(open(secrets_path, "rb"))["MONGODB_URI"]

In [15]:
# Test MongoDB connection
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [19]:
# Defining functions to extract data from Elhub API 
import requests
import pandas as pd
import time

# API endpoint
BASE_URL = "https://api.elhub.no/energy-data/v0/price-areas"

def month_ranges(year: int):
    """Yield (start_date, end_date) for each month of given year as YYYY-MM-DD strings."""
    months = pd.date_range(f"{year}-01-01", f"{year}-12-31", freq="MS")
    for start in months:
        end = (start + pd.offsets.MonthEnd(1)).normalize()
        yield start.date().isoformat(), end.date().isoformat()


def fetch_elhub(dataset: str, years, list_key: str) -> pd.DataFrame:
    """
    Fetch data from Elhub for a dataset across multiple years.
    - dataset: API dataset name
    - years: list of years
    - list_key: the key inside 'attributes' containing the hourly records
    """
    all_records = []

    for year in years:
        print(f"Fetching {dataset} for year {year} ...")
        
        for start_date, end_date in month_ranges(year):

            params = {
                "dataset": dataset,
                "startDate": start_date,
                "endDate": end_date,
            }

            resp = requests.get(BASE_URL, params=params, timeout=60)
            resp.raise_for_status()
            payload = resp.json()

            # Extract rows from the nested attributes object
            for item in payload.get("data", []):
                rows = item.get("attributes", {}).get(list_key, [])
                all_records.extend(rows)
            
            # To keep within API rate limits
            time.sleep(0.2)
            
    print(f"Fetched all data in dataset {dataset}. Total records: {len(all_records)}")
    
    return pd.DataFrame(all_records)

In [20]:
# Extracting production data from Elhub API
df_prod_raw = fetch_elhub(
    dataset="PRODUCTION_PER_GROUP_MBA_HOUR",
    years=[2022, 2023, 2024], 
    list_key="productionPerGroupMbaHour",
)
df_prod_raw.shape

Fetching PRODUCTION_PER_GROUP_MBA_HOUR for year 2022 ...
Fetching PRODUCTION_PER_GROUP_MBA_HOUR for year 2023 ...
Fetching PRODUCTION_PER_GROUP_MBA_HOUR for year 2024 ...
Fetched all data in dataset PRODUCTION_PER_GROUP_MBA_HOUR. Total records: 636025


(636025, 6)

In [21]:
# Extracting consumption data from Elhub API
df_cons_raw = fetch_elhub(
    dataset="CONSUMPTION_PER_GROUP_MBA_HOUR",
    years=[2021, 2022, 2023, 2024],
    list_key="consumptionPerGroupMbaHour",
)
df_cons_raw.shape

Fetching CONSUMPTION_PER_GROUP_MBA_HOUR for year 2021 ...
Fetching CONSUMPTION_PER_GROUP_MBA_HOUR for year 2022 ...
Fetching CONSUMPTION_PER_GROUP_MBA_HOUR for year 2023 ...
Fetching CONSUMPTION_PER_GROUP_MBA_HOUR for year 2024 ...
Fetched all data in dataset CONSUMPTION_PER_GROUP_MBA_HOUR. Total records: 847800


(847800, 7)

In [30]:
# Function for cleaning Elhub data
import pandas as pd

def clean_elhub_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only the required columns and clean time / numeric types.
    Works for both production (productionGroup) and consumption (consumptionGroup).
    """

    # Detect which group column is present
    if "productionGroup" in df.columns:
        group_col = "productionGroup"
    elif "consumptionGroup" in df.columns:
        group_col = "consumptionGroup"
    else:
        raise ValueError("Expected 'productionGroup' or 'consumptionGroup' in dataframe")

    # Keep only relevant columns
    cols = ["priceArea", group_col, "startTime", "quantityKwh"]
    df = df[cols].copy()

    # Convert startTime to UTC and drop timezone
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True, errors="coerce")
    df["startTime"] = df["startTime"].dt.tz_localize(None)

    # Ensure quantityKwh is numeric
    df["quantityKwh"] = pd.to_numeric(df["quantityKwh"], errors="coerce")

    # Drop rows with missing critical values
    df = df.dropna(subset=["startTime", "quantityKwh"])

    # Sort and reset index
    df = df.sort_values(["priceArea", group_col, "startTime"]).reset_index(drop=True)

    return df

In [ ]:
# Showing cleaned production data
df_prod_clean = clean_elhub_df(df_prod_raw)
df_prod_clean

,priceArea,productionGroup,startTime,quantityKwh
0,NO1,hydro,2021-12-31 23:00:00,1291422.4
1,NO1,hydro,2022-01-01 00:00:00,1246209.4
2,NO1,hydro,2022-01-01 01:00:00,1271757.0
3,NO1,hydro,2022-01-01 02:00:00,1204251.8
4,NO1,hydro,2022-01-01 03:00:00,1202086.9
...,...,...,...,...
636020,NO5,wind,2024-12-30 18:00:00,0.0
636021,NO5,wind,2024-12-30 19:00:00,0.0
636022,NO5,wind,2024-12-30 20:00:00,0.0
636023,NO5,wind,2024-12-30 21:00:00,0.0


In [ ]:
# Showing cleaned consumption data
df_cons_clean = clean_elhub_df(df_cons_raw)
df_cons_clean

,priceArea,consumptionGroup,startTime,quantityKwh
0,NO1,cabin,2020-12-31 23:00:00,177071.56
1,NO1,cabin,2021-01-01 00:00:00,171335.12
2,NO1,cabin,2021-01-01 01:00:00,164912.02
3,NO1,cabin,2021-01-01 02:00:00,160265.77
4,NO1,cabin,2021-01-01 03:00:00,159828.69
...,...,...,...,...
847795,NO5,tertiary,2024-12-30 18:00:00,393986.80
847796,NO5,tertiary,2024-12-30 19:00:00,386786.88
847797,NO5,tertiary,2024-12-30 20:00:00,368434.84
847798,NO5,tertiary,2024-12-30 21:00:00,345032.94


In [35]:
# Create keyspace and tables in Cassandra

from cassandra.cluster import Cluster

KEYSPACE   = "elhub2021"
PROD_TABLE = "prod_by_group_hour"
CONS_TABLE = "cons_by_group_hour"

# Connect to Cassandra
# cluster = Cluster(["localhost"], port=9042)
cluster = Cluster(['127.0.0.1'], port=9042)
session = cluster.connect()

# Keyspace
session.execute(f"""
    CREATE KEYSPACE IF NOT EXISTS {KEYSPACE}
    WITH replication = {{ 'class': 'SimpleStrategy', 'replication_factor': '1' }}
""")

session.set_keyspace(KEYSPACE)

# Table for production
session.execute(f"""
    CREATE TABLE IF NOT EXISTS {PROD_TABLE} (
        pricearea       TEXT,
        productiongroup TEXT,
        starttime       TIMESTAMP,
        quantitykwh     DOUBLE,
        PRIMARY KEY ((pricearea, productiongroup), starttime)
    )
    WITH CLUSTERING ORDER BY (starttime ASC)
""")

# Table for consumption
session.execute(f"""
    CREATE TABLE IF NOT EXISTS {CONS_TABLE} (
        pricearea        TEXT,
        consumptiongroup TEXT,
        starttime        TIMESTAMP,
        quantitykwh      DOUBLE,
        PRIMARY KEY ((pricearea, consumptiongroup), starttime)
    )
    WITH CLUSTERING ORDER BY (starttime ASC)
""")

print("Keyspace and tables are ready.")

Keyspace and tables are ready.


In [ ]:
# Write production data to Cassandra using Spark  

# Prodcution data for 2022-2024
df_prod_spark = spark.createDataFrame(df_prod_clean).toDF(
    "pricearea", "productiongroup", "starttime", "quantitykwh"
)

(
    df_prod_spark.write
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace=KEYSPACE, table=PROD_TABLE)
    .mode("append")          # viktig: append til 2021-data
    .save()
)

print("Production data written to Cassandra.")

Production data written to Cassandra.


In [36]:
# Write consumption data to Cassandra using Spark

# Consumption data for 2021-2024
df_cons_spark = spark.createDataFrame(df_cons_clean).toDF(
    "pricearea", "consumptiongroup", "starttime", "quantitykwh"
)

(
    df_cons_spark.write
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace=KEYSPACE, table=CONS_TABLE)
    .mode("append")
    .save()
)

print("Consumption data written to Cassandra.")

Consumption data written to Cassandra.


In [38]:
# Read production data from Cassandra
df_prod_read = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace=KEYSPACE, table=PROD_TABLE)
    .load()
)

df_prod_read.printSchema()
df_prod_read.show(5, truncate=False)

root
 |-- pricearea: string (nullable = false)
 |-- starttime: timestamp (nullable = true)
 |-- productiongroup: string (nullable = true)
 |-- quantitykwh: double (nullable = true)

+---------+-------------------+---------------+-----------+
|pricearea|starttime          |productiongroup|quantitykwh|
+---------+-------------------+---------------+-----------+
|NO5      |2020-12-31 23:00:00|hydro          |4068096.5  |
|NO5      |2020-12-31 23:00:00|other          |0.0        |
|NO5      |2020-12-31 23:00:00|solar          |3.72       |
|NO5      |2020-12-31 23:00:00|thermal        |77742.0    |
|NO5      |2021-01-01 00:00:00|hydro          |4104306.0  |
+---------+-------------------+---------------+-----------+
only showing top 5 rows



In [39]:
# Read consumption data from Cassandra
df_cons_read = (
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace=KEYSPACE, table=CONS_TABLE)
    .load()
)

df_cons_read.printSchema()
df_cons_read.show(5, truncate=False)

root
 |-- pricearea: string (nullable = false)
 |-- consumptiongroup: string (nullable = false)
 |-- starttime: timestamp (nullable = true)
 |-- quantitykwh: double (nullable = true)

+---------+----------------+-------------------+-----------+
|pricearea|consumptiongroup|starttime          |quantitykwh|
+---------+----------------+-------------------+-----------+
|NO1      |cabin           |2020-12-31 23:00:00|177071.56  |
|NO1      |cabin           |2021-01-01 00:00:00|171335.12  |
|NO1      |cabin           |2021-01-01 01:00:00|164912.02  |
|NO1      |cabin           |2021-01-01 02:00:00|160265.77  |
|NO1      |cabin           |2021-01-01 03:00:00|159828.69  |
+---------+----------------+-------------------+-----------+
only showing top 5 rows



In [40]:
# Insert production data from Cassandra into MongoDB
from pymongo import MongoClient
from pymongo.server_api import ServerApi

df_prod_pd = df_prod_read.toPandas()
prod_records = df_prod_pd.to_dict(orient="records")

# Connect to MongoDB
client = MongoClient(uri, server_api=ServerApi('1'))
client.admin.command("ping")

db = client["elhub2021"]
prod_col = db["production_per_group_hour"]

prod_col.insert_many(prod_records)
print(f"Inserted {len(prod_records)} production rows from Cassandra into MongoDB.")

client.close()

Inserted 844199 production rows from Cassandra into MongoDB.


In [41]:
# Insert consumption data from Cassandra into MongoDB
from pymongo import MongoClient
from pymongo.server_api import ServerApi

df_cons_pd = df_cons_read.toPandas()
cons_records = df_cons_pd.to_dict(orient="records")

# Connect to MongoDB
client = MongoClient(uri, server_api=ServerApi('1'))
client.admin.command("ping")

db = client["elhub2021"]
cons_col = db["consumption_per_group_hour"]

cons_col.insert_many(cons_records)
print(f"Inserted {len(cons_records)} consumption rows from Cassandra into MongoDB.")

client.close()

Inserted 847725 consumption rows from Cassandra into MongoDB.


## Streamlit app, refactoring
- Plotting
    - Exchange static plots (matplotlib/seaborn) with dynamic plots (Plotly, Streamlit, Bokeh, ...).
    - Use AI to help you translate old plot code if you want to save time.
- Structure and navigation
    - Find a way of ordering the pages that makes logical sense, e.g., based on the type of tasks (explorative, anomalies, predictive), the source of the data, regional vs local data, or similar.
    - Before deciding on the structure, check which new elements are to be included below. Use pen and paper, PowerPoint, or another tool to organize your content before changing your code.
    - Let the menubar reflect these changes, possibly including grouping, headers, or similar. You can, for instance, have one common front page pointing to two or three main analysis pages, which have their own menus pointing to single pages. If you group pages like this, consider colour themes or other effects to emphasize which group is active.
    - Regarding user-friendliness, for some pages it is natural to rely on selection on a main page, for other pages, local selections make more sense (e.g., year, location, ...).
    - Should we let the LineChartColumn page go -- or maybe join it with other data overview stuff?-

## Streamlit app, new elements

- Map and selectors
  - Plotly or Folio map with GeoJSON-based overlay of Price areas NO1–NO5.
    - Download GeoJSON data for Price areas from https://temakart.nve.no/tema/nettanlegg by finding "NVE Elspot områder" and "ElSpot_omraade", selecting all areas, and exporting to GeoJSON format.
    - Show outlines of the Price areas.
    - Let any clicked coordinate be stored in the app and marked in the plot. Also, mark the chosen Price area with a different outline.
    - See the bonus section below for extra possibilities regarding municipalities.
  - Let the user choose an energy production/energy consumption group and a time interval (in days).
  - Colour the price areas transparently (e.g., choropleth) according to the mean values over the time interval for the chosen production/consumption.

- Snow drift calculation and illustration
  - Copy and edit the relevant parts of the supplied file `Snow_drift.py`.
  - Define a year as 1 July in the selected year to 30 June the following year.
  - Calculate snow drift per year in a selected range of years and plot.
    - Let the user choose the year range.
    - Use the coordinates chosen on the map page (gracefully refuse to plot/calculate if no selection has been made).
  - Plot the corresponding wind rose (see `Snow_drift.py`).
  - See the bonus section below for extra possibilities regarding monthly snow drift.

- Meteorology and energy production
  - Transform the Sliding Window Correlation with selectable lag and window length into a Streamlit equivalent with Plotly graphics (or equivalent).
  - Include one selector for meteorological properties and one selector for energy production and consumption.
  - After implementation, test the controls and check for changes in correlations under normal conditions and during/after extreme weather events. Report findings in the Jupyter Notebook log.

- Forecasting of energy production and consumption
  - Create an interface for SARIMAX where all relevant parameters are selectable.
  - Let the user choose the timeframe for the training data and the forecast horizon.
  - Forecast a user-selected property from energy production/consumption with dynamic forecasting, including any selected exogenous variables.
  - Plot the results, including confidence intervals.
  - See the bonus section below regarding weather properties.


## Streamlit app, bonus content. Select at least one task from this list. Report in the log which one you chose.

- Waiting time
  - Use progress bars, spinners, or similar to indicate work in progress.
  - Cache everything that is possible to cache.

- Error handling
  - Try to incorporate checks in the app that handle missing data connections (API and database) and NaN/missing values in the data.
  - Catch the errors and give useful feedback instead of crashing or giving cryptic error messages.

- Map page
  - Let the map page display municipalities when magnified beyond a natural threshold and price areas otherwise.
  - GeoJSON data: http://kartkatalog.geonorge.no using the EUREF 89 Geografisk (ETRS 89) 2d format.
  - Possible strategy: Plot both grids (municipality and price area) and set minzoom/maxzoom for each to control their visibility.

- Snow drift
  - Calculate the monthly snow drift and plot it together with the yearly snow drift.

- Forecasting
  - Add weather properties to the list of exogenous variables and download when needed.

- Elevation
  - open-meteo also has an elevation API. This can be used to add elevation information to selected points in the main map or to make a separate elevation map.


## Work Log:

**Jupyter Notebook:**
- Decided that the best choice would be to copy the code from assignment 2 and modify it for this one, so that's what I did. 
- Connection to Cassandra og MongoDB went pretty seemless and was copied directly into this notebook. 
- When fetching the data from Elhub API I decided that creating one generalized function would be best so I could use it to fetch both production and consumption data. 
- Tested the function by extracting the data and creating a df for each. 
- Then continuing to cleaning the data. Where I created a function for that also. Then showed the cleaned data.
- Next up was to do the Cassandra-Spark-Mongo part. Was quite identical to assignment 2, apart from setting up a new table for consumption data inside the existing keyspace.
- Everything went seemless and was inserted into MongoDB successfully. 
- (Keyspace name elhub2021 might be a bit misleading, but decided to keep the same name regardless)

**Streamlit app, refactoring:**
- In the previous assignment I had forgotten to use plotly instead of matplotlib when I created page New A and page New B, so I changed the plots to use plotly instead. 
- Then when it came to planning I first stopped to remember what I had already done in previous assignments and to remember what my streamlit app looks like now. Then I went ahead and tried to understand what parts should be added to my app in the next part. 
- After getting the understanding I decided on the structure of the menu and the order of the pages. 

**Streamlit app, new elements:**

Map and selectors:
- For this part I first added a new cache function for geojson file. 
- On the page we built controls to choose data type (production/consumption), group, and time interval, and then computed mean kWh per price area.
- We used Folium to draw the NO1–NO5 polygons as a choropleth map, highlighting the selected price area from the Production Explorer page.
- Finally, we enabled map clicks and stored the clicked coordinate in st.session_state["map_coord"] for use on the snow-drift page.

Snow drift:
- For the snow drift page, I reused the coordinate stored by the map page (from st.session_state["map_coord"]) as the input location. 
- I then downloaded hourly weather data from the Open-Meteo archive for all calendar years needed to cover the selected July–June seasons and prepared a DataFrame with a season column. 
- Using the existing functions in src.Snow_drift, I computed yearly snow transport (Qt) for each season and the average directional sector values. 
- Finally, I visualised the results in Streamlit with Plotly: a bar chart of yearly Qt in tonnes per metre and a polar wind-rose plot, and added basic error handling if no coordinate or data is available.

Meteorology and energy production: 
- On the left side, we added controls to select price area, year, weather variable, energy type (production/consumption), group, lag, and window length.
- We load weather data from Open-Meteo and production/consumption data from MongoDB using the shared data_loader functions.
- The code aligns the two time series, applies the chosen lag, and computes a rolling (sliding window) Pearson correlation.
- On the right side, we show Plotly charts for the time series and the rolling correlation, together with a short text summary of the settings and overall correlation.

- **To test the controls** I selected year 2024 and price area NO1, and analysed the correlation between precipitation and hydro production.
- Precipitation is generally higher during spring and summer, while hydro production appears very spiky and does not visually follow the precipitation pattern.
- The sliding window correlation stays close to zero over the year, which indicates a weak linear relationship for this combination.
- On 21 August there is a clear extreme rainfall event, but the rolling correlation does not show any strong change around this period.
- This suggests that other factors such as reservoir management, prices and operational constraints are more important drivers of short-term hydro production than hourly precipitation alone.

Energy forecasting:
- Built a left control panel where the user selects dataset type, price area, group, frequency (hourly/daily), training period, SARIMAX orders and forecast horizon.
- Converted the selected data to the chosen frequency, limited the training window to a reasonable size, and prepared the time series for modeling.
- Fitted a SARIMAX model on the training data and generated a dynamic forecast for the chosen horizon.
- Plotted training data, in-sample fitted values, forecast and confidence interval with Plotly, and displayed a simple training RMSE below the plot.

**Streamlit app, bonus content:**
- I chose the waiting time bonus. Didn't require any extra work, because I've incorporated cache from the beginning. 

## AI usage:

- Have used ChatGPT with a rotation between 5.1 regular and thinking version depending on the task. Normally more difficult tasks required more thinking power - like debugging. 

**Jupyter Notebook:**
- For fetching the elhub API data I used AI to rewrite the code for fetching the data to be more generalized. This way I could use to same function for the consumption and production data. 
- Had to debug connection to cassandra and connect explicitly to IPv4 instead of localhost. It worked. 
- Used AI to rewrite code so I could use it on the consumption data like I did with the production data. 

**Streamlit app, refactoring:**
- To change the plotting from matplotlib to plotly I gave the code that previously used plt and told AI to rewrite the code to using plotly instead. 
- In this part I used AI as a sparring partner to decide the layout and architechture of what my app should look like and order of pages. 

**Streamlit app, new elements:**

Map and selectors:
- I used AI to work out a specific stepwise plan for how and what to include in the code. 
- Then I worked back and forth to create the code with AI and debug along the way when issues arose. 

Snow drift:
- Yet again used AI to come up with a plan together for what to do stepwise and to make sure I don't miss any steps. 
- Used AI to write most of the code, but did my own changes and adjustments that didn't make sense. 

Meteorology and energy production: 
- Like previously I made a plan with on how to structure the code and what to include.
- We sparred together to find a good solution UI-wise. 
- Lots of the code were directly written by AI, but with me designing the structure and user experience.  

Energy forecasting:
- Here I used AI especially on debugging and changing the code such that the Sarimax forecasting gave expected results. 

**Streamlit app, bonus content:**

Waiting time: 
- Nothing here, because I've used cache all througout the project. 